# LN5: Image Representation and Spatial Filtering

This Colab notebook follows the two main topics in **LN_05 Image & Filtering**:

1. Digital images: sampling, quantization, color, point processing, histograms, intensity profiles, and image noise.
2. Spatial filtering: linear filters, padding, correlation and convolution, Gaussian and median filters, sharpening, and image derivatives.

Run the cells from top to bottom. Upload one image when prompted. The same image is used throughout the notebook.


## Learning outcomes

After completing the notebook, a student should be able to:

- Represent an image as a sampled and quantized array.
- Inspect RGB channels, grayscale intensity, histograms, and row profiles.
- Simulate common noise models and explain why averaging reduces random noise.
- Calculate 1D and 2D filter responses by hand and in Python.
- Compare padding modes, correlation, and convolution.
- Apply box, Gaussian, median, sharpening, and derivative filters.
- Explain linearity, shift invariance, separability, and computational cost.


## Lecture coverage map

- Slides 4 to 18: image functions, digitization, sampling, resolution, quantization, grayscale, and RGB channels.
- Slides 19 to 34: noise models, point processing, histograms, intensity profiles, and averaging repeated captures.
- Slides 35 to 62: neighborhood filtering, 1D and 2D calculations, output size, padding, identity, shift, box, and sharpening kernels.
- Slides 66 to 95: linear shift-invariant systems, correlation, convolution, Gaussian filters, separability, and runtime.
- Slides 96 to 108: outliers, median filtering, nonlinearity, and unsharp masking.
- Slides 109 to 119: derivative filters, gradient magnitude, and numerical padding exercises.


## 0. Setup and image upload

Upload a JPG, JPEG, or PNG image. For faster experiments, the notebook limits the longest image side to 900 pixels while preserving the aspect ratio.


In [ ]:
import io
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["image.cmap"] = "gray"

def show_images(images, titles, cols=3, figsize=None, cmap=None):
    rows = math.ceil(len(images) / cols)
    if figsize is None:
        figsize = (5 * cols, 4 * rows)
    plt.figure(figsize=figsize)
    for i, (image, title) in enumerate(zip(images, titles), start=1):
        ax = plt.subplot(rows, cols, i)
        ax.imshow(image, cmap=cmap if image.ndim == 2 else None, vmin=0 if image.ndim == 2 else None, vmax=255 if image.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

def to_uint8(image):
    return np.clip(np.rint(image), 0, 255).astype(np.uint8)

try:
    from google.colab import files
    uploaded = files.upload()
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab to use the upload control.") from exc

if not uploaded:
    raise ValueError("No image was uploaded.")

filename, file_bytes = next(iter(uploaded.items()))
rgb = np.array(Image.open(io.BytesIO(file_bytes)).convert("RGB"))

max_side = 900
h, w = rgb.shape[:2]
if max(h, w) > max_side:
    scale = max_side / max(h, w)
    rgb = cv2.resize(rgb, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)

print(f"Loaded: {filename}")
print(f"Shape: {rgb.shape} | dtype: {rgb.dtype} | range: [{rgb.min()}, {rgb.max()}]")
show_images([rgb], ["Uploaded RGB image"], cols=1, figsize=(8, 6))


# Part I: Digital images

An image is a function P(x, y). Digitization makes the spatial domain discrete through **sampling** and makes the intensity range discrete through **quantization**. A grayscale pixel normally stores one value. An RGB pixel stores red, green, and blue values.


## 1. RGB channels and grayscale intensity

The lecture uses the weighted grayscale conversion

g = 0.2989R + 0.5870G + 0.1140B.

The green channel receives the largest weight because this approximation follows human luminance sensitivity.


In [ ]:
R = rgb[:, :, 0]
G = rgb[:, :, 1]
B = rgb[:, :, 2]
gray_float = 0.2989 * R + 0.5870 * G + 0.1140 * B
gray = to_uint8(gray_float)

show_images(
    [rgb, R, G, B, gray],
    ["RGB", "Red channel", "Green channel", "Blue channel", "Weighted grayscale"],
    cols=3,
    figsize=(15, 9),
)

row, col = gray.shape[0] // 2, gray.shape[1] // 2
print(f"Center pixel RGB = {rgb[row, col].tolist()}")
print(f"Calculated grayscale intensity = {gray_float[row, col]:.2f}")


## 2. Sampling and spatial resolution

Downsampling uses fewer spatial samples. Upscaling cannot recover the missing samples. Nearest-neighbor enlargement makes the pixel grid visible.


In [ ]:
h, w = rgb.shape[:2]
half = cv2.resize(rgb, (max(1, w // 2), max(1, h // 2)), interpolation=cv2.INTER_AREA)
quarter = cv2.resize(rgb, (max(1, w // 4), max(1, h // 4)), interpolation=cv2.INTER_AREA)
half_up = cv2.resize(half, (w, h), interpolation=cv2.INTER_NEAREST)
quarter_up = cv2.resize(quarter, (w, h), interpolation=cv2.INTER_NEAREST)

show_images(
    [rgb, half, quarter, half_up, quarter_up],
    [f"Original {w} x {h}", "2x smaller", "4x smaller", "2x smaller then upscaled", "4x smaller then upscaled"],
    cols=3,
    figsize=(15, 9),
)


## 3. Intensity quantization

For b bits, the number of available intensity levels is 2^b. This example compares the original 8-bit image with 4-bit, 3-bit, and 1-bit representations.


In [ ]:
def quantize_uint8(image, bits):
    levels = 2 ** bits
    indices = np.floor(image.astype(np.float32) * levels / 256.0)
    indices = np.clip(indices, 0, levels - 1)
    if levels == 1:
        return np.zeros_like(image)
    return to_uint8(indices * 255.0 / (levels - 1))

quantized = [quantize_uint8(rgb, bits) for bits in [8, 4, 3, 1]]
show_images(quantized, ["8-bit: 256 levels", "4-bit: 16 levels", "3-bit: 8 levels", "1-bit: 2 levels"], cols=2)


## 4. Point processing

Point operations transform each pixel independently. The lecture examples reduce contrast, increase contrast with saturation, add 20 to the brightness, convert to grayscale, and reflect the image horizontally.


In [ ]:
low_contrast = to_uint8(rgb.astype(np.float32) / 2)
high_contrast = to_uint8(rgb.astype(np.float32) * 2)
brighter = to_uint8(rgb.astype(np.float32) + 20)
horizontal_reflection = np.fliplr(rgb)

show_images(
    [rgb, low_contrast, high_contrast, brighter, gray, horizontal_reflection],
    ["Original", "Low contrast: f / 2", "High contrast: 2f with saturation", "Brightness: f + 20", "Grayscale", "Horizontal reflection: f(-x, y)"],
    cols=3,
    figsize=(15, 9),
)


## 5. Histogram and row intensity profiles

A histogram counts how often each intensity occurs. A row profile shows how intensity changes along one selected row.


In [ ]:
rows_to_plot = [gray.shape[0] // 3, 2 * gray.shape[0] // 3]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].imshow(gray, cmap="gray", vmin=0, vmax=255)
for r in rows_to_plot:
    axes[0].axhline(r, linewidth=1.5, label=f"row {r}")
axes[0].set_title("Selected rows")
axes[0].legend()
axes[0].axis("off")

axes[1].hist(gray.ravel(), bins=256, range=(0, 256), color="black")
axes[1].set_title("Grayscale histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Pixel count")

for r in rows_to_plot:
    axes[2].plot(gray[r, :], label=f"row {r}")
axes[2].set_title("Intensity profiles")
axes[2].set_xlabel("Column")
axes[2].set_ylabel("Intensity")
axes[2].set_ylim(0, 255)
axes[2].legend()
plt.tight_layout()
plt.show()


## 6. Image noise models

The lecture presents additive noise, multiplicative noise, Gaussian noise, uniform noise, and salt-and-pepper noise. A fixed random seed makes the experiment repeatable.


In [ ]:
rng = np.random.default_rng(441)

def add_gaussian_noise(image, sigma=20):
    noise = rng.normal(0, sigma, image.shape)
    return to_uint8(image.astype(np.float32) + noise)

def add_uniform_noise(image, amplitude=30):
    noise = rng.uniform(-amplitude, amplitude, image.shape)
    return to_uint8(image.astype(np.float32) + noise)

def add_multiplicative_noise(image, sigma=0.18):
    factor = 1.0 + rng.normal(0, sigma, image.shape)
    return to_uint8(image.astype(np.float32) * factor)

def add_salt_pepper_noise(image, probability=0.08):
    noisy = image.copy()
    random_map = rng.random(image.shape[:2])
    noisy[random_map < probability / 2] = 0
    noisy[(random_map >= probability / 2) & (random_map < probability)] = 255
    return noisy

gaussian_noisy = add_gaussian_noise(rgb)
uniform_noisy = add_uniform_noise(rgb)
multiplicative_noisy = add_multiplicative_noise(rgb)
salt_pepper_noisy = add_salt_pepper_noise(rgb)

show_images(
    [rgb, gaussian_noisy, uniform_noisy, multiplicative_noisy, salt_pepper_noisy],
    ["Original", "Additive Gaussian", "Additive uniform", "Multiplicative", "Salt and pepper"],
    cols=3,
    figsize=(15, 9),
)


## 7. Noise reduction by averaging several images

For a still scene with independent zero-mean noise, averaging several captures reduces noise. The standard deviation falls approximately in proportion to 1 / sqrt(N).


In [ ]:
clean = rgb.astype(np.float32)
averages = []
rmse_values = []
counts = [1, 4, 16, 64]

for count in counts:
    stack = np.stack([clean + rng.normal(0, 25, clean.shape) for _ in range(count)])
    average = np.clip(stack.mean(axis=0), 0, 255)
    averages.append(average.astype(np.uint8))
    rmse_values.append(np.sqrt(np.mean((average - clean) ** 2)))

show_images(averages, [f"Average of {n} noisy image(s)\nRMSE = {e:.2f}" for n, e in zip(counts, rmse_values)], cols=2)
print("RMSE values:", dict(zip(counts, np.round(rmse_values, 3))))


# Part II: Spatial filtering

Spatial filtering computes each output pixel from a local neighborhood. Linear filters use a weighted sum of the neighborhood values.


## 8. Lecture 1D moving-average calculation

Signal: [10, 12, 9, 11, 10, 11, 12]

Filter: [1/3, 1/3, 1/3]

The first output is (10 + 12 + 9) / 3 = 10.33. The filter then moves one position at a time.


In [ ]:
signal = np.array([10, 12, 9, 11, 10, 11, 12], dtype=float)
average_kernel_1d = np.ones(3) / 3
output_1d = np.correlate(signal, average_kernel_1d, mode="valid")

for i, value in enumerate(output_1d):
    window = signal[i:i + 3]
    print(f"y[{i}] = ({window[0]:.0f} + {window[1]:.0f} + {window[2]:.0f}) / 3 = {value:.2f}")


## 9. Manual 2D correlation

This implementation exposes the arithmetic used by a 2D linear filter. It supports valid output and same-size output with constant, symmetric, reflect, or wrap padding.


In [ ]:
def correlate2d_manual(image, kernel, mode="same", padding="constant", constant_value=0):
    image = np.asarray(image, dtype=float)
    kernel = np.asarray(kernel, dtype=float)
    kh, kw = kernel.shape
    if kh % 2 == 0 or kw % 2 == 0:
        raise ValueError("Use odd kernel dimensions for this teaching function.")

    if mode == "valid":
        padded = image
    elif mode == "same":
        py, px = kh // 2, kw // 2
        pad_mode = {"constant": "constant", "symmetric": "symmetric", "reflect": "reflect", "wrap": "wrap"}[padding]
        kwargs = {"constant_values": constant_value} if pad_mode == "constant" else {}
        padded = np.pad(image, ((py, py), (px, px)), mode=pad_mode, **kwargs)
    else:
        raise ValueError("mode must be 'same' or 'valid'")

    oh = padded.shape[0] - kh + 1
    ow = padded.shape[1] - kw + 1
    output = np.empty((oh, ow), dtype=float)
    for y in range(oh):
        for x in range(ow):
            patch = padded[y:y + kh, x:x + kw]
            output[y, x] = np.sum(patch * kernel)
    return output

small_image = np.arange(1, 31, dtype=float).reshape(5, 6)
box3 = np.ones((3, 3), dtype=float) / 9
manual_valid = correlate2d_manual(small_image, box3, mode="valid")

print("Input shape:", small_image.shape)
print("3x3 valid output shape:", manual_valid.shape)
print("Number of valid positions:", manual_valid.size)
print(manual_valid)


## 10. Output size and boundary handling

- Valid: compute only where the kernel fits completely.
- Same: pad the input so the output keeps the input size.
- Full: include every partial overlap.

Common padding choices include zeros, symmetric extension, reflection, and circular wrapping.


In [ ]:
from scipy.signal import correlate2d

toy = np.array([
    [10, 20, 30, 40],
    [15, 25, 35, 45],
    [18, 28, 38, 48],
], dtype=float)

valid = correlate2d(toy, box3, mode="valid")
same_zero = correlate2d_manual(toy, box3, mode="same", padding="constant")
same_symmetric = correlate2d_manual(toy, box3, mode="same", padding="symmetric")
same_wrap = correlate2d_manual(toy, box3, mode="same", padding="wrap")
full = correlate2d(toy, box3, mode="full", boundary="fill", fillvalue=0)

print("Input shape:", toy.shape)
print("Valid shape:", valid.shape, "Same shape:", same_zero.shape, "Full shape:", full.shape)
print(f"Top-left with zero padding      = {same_zero[0, 0]:.3f}  (lecture: 7.778)")
print(f"Top-left with symmetric padding = {same_symmetric[0, 0]:.3f}  (lecture: 15.000)")
print(f"Top-left with circular padding  = {same_wrap[0, 0]:.3f}  (lecture: 27.667)")

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, data, title in zip(axes, [toy, same_zero, same_symmetric, same_wrap], ["Input", "Zero padding", "Symmetric padding", "Circular padding"]):
    ax.imshow(data, cmap="gray")
    ax.set_title(title)
    for (y, x), value in np.ndenumerate(data):
        ax.text(x, y, f"{value:.1f}", ha="center", va="center", color="red")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 11. Identity, shift, box, and sharpening filters

The identity kernel preserves the image. Moving the single 1 away from the center shifts the image. A box filter blurs. A sharpening kernel can be formed as 2 times the identity minus the box filter.


In [ ]:
identity = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=float)
shift_left = np.array([[0, 0, 0], [0, 0, 1], [0, 0, 0]], dtype=float)
shift_down = np.array([[0, 1, 0], [0, 0, 0], [0, 0, 0]], dtype=float)
sharpen_kernel = 2 * identity - box3

def filter_rgb(image, kernel, border=cv2.BORDER_REFLECT_101):
    channels = [cv2.filter2D(image[:, :, c].astype(np.float32), -1, kernel, borderType=border) for c in range(3)]
    return to_uint8(np.stack(channels, axis=2))

identity_result = filter_rgb(rgb, identity)
left_result = filter_rgb(rgb, shift_left, border=cv2.BORDER_CONSTANT)
down_result = filter_rgb(rgb, shift_down, border=cv2.BORDER_CONSTANT)
box_result = filter_rgb(rgb, box3)
sharp_result = filter_rgb(rgb, sharpen_kernel)

print("Sharpening kernel = 2 x identity - box")
print(np.round(sharpen_kernel, 3))
show_images(
    [rgb, identity_result, left_result, down_result, box_result, sharp_result],
    ["Original", "Identity", "Shift left by 1", "Shift down by 1", "3x3 box blur", "2I - box sharpening"],
    cols=3,
    figsize=(15, 9),
)


## 12. Linearity and shift invariance

A linear filter satisfies superposition and scaling. A shift-invariant filter gives the same result when shifting before or after filtering, apart from boundary pixels introduced by padding.


In [ ]:
f1 = np.ones((3, 3), dtype=float) / 9
f2 = identity.copy()
image_float = gray.astype(float)

left_side = cv2.filter2D(image_float, -1, f1 + f2, borderType=cv2.BORDER_REFLECT_101)
right_side = cv2.filter2D(image_float, -1, f1, borderType=cv2.BORDER_REFLECT_101) + cv2.filter2D(image_float, -1, f2, borderType=cv2.BORDER_REFLECT_101)
print("Maximum linearity error:", np.max(np.abs(left_side - right_side)))

shifted_input = np.roll(image_float, shift=8, axis=1)
filter_then_shift = np.roll(ndimage.correlate(image_float, f1, mode="wrap"), shift=8, axis=1)
shift_then_filter = ndimage.correlate(shifted_input, f1, mode="wrap")
print("Maximum shift-invariance error with circular boundary:", np.max(np.abs(filter_then_shift - shift_then_filter)))


## 13. Cross-correlation versus convolution

Cross-correlation uses the kernel in its original orientation. Convolution flips the kernel horizontally and vertically first. Symmetric kernels such as box and Gaussian kernels give the same result in both operations.


In [ ]:
asymmetric_kernel = np.array([
    [0, 0, 1],
    [0, 0, 2],
    [0, 0, 3],
], dtype=float)

correlation = cv2.filter2D(gray.astype(np.float32), -1, asymmetric_kernel, borderType=cv2.BORDER_REFLECT_101)
flipped_kernel = np.flip(asymmetric_kernel, axis=(0, 1))
convolution = cv2.filter2D(gray.astype(np.float32), -1, flipped_kernel, borderType=cv2.BORDER_REFLECT_101)

print("Original kernel:\n", asymmetric_kernel)
print("Flipped convolution kernel:\n", flipped_kernel)
show_images([to_uint8(correlation), to_uint8(convolution)], ["Cross-correlation", "Convolution"], cols=2)


## 14. Gaussian filters and filter size

A 2D Gaussian kernel weights nearby pixels more strongly. A practical odd kernel size is approximately 6 sigma + 1 so that the kernel covers about three standard deviations on each side.


In [ ]:
def gaussian_kernel_2d(sigma, size=None):
    if size is None:
        size = int(2 * math.ceil(3 * sigma) + 1)
    if size % 2 == 0:
        size += 1
    radius = size // 2
    x = np.arange(-radius, radius + 1, dtype=float)
    xx, yy = np.meshgrid(x, x)
    kernel = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return kernel / kernel.sum()

sigmas = [1, 3, 7]
gaussian_results = []
for sigma in sigmas:
    kernel = gaussian_kernel_2d(sigma)
    gaussian_results.append(filter_rgb(rgb, kernel))
    print(f"sigma={sigma}: kernel shape={kernel.shape}, sum={kernel.sum():.6f}")

show_images([rgb] + gaussian_results, ["Original"] + [f"Gaussian sigma={s}" for s in sigmas], cols=2)


## 15. Gaussian separability and computational cost

A 2D Gaussian equals a vertical 1D Gaussian followed by a horizontal 1D Gaussian. For an N by N image and an M by M filter:

- Direct 2D filtering costs approximately N^2 M^2 multiply-adds.
- Separable filtering costs approximately 2 N^2 M multiply-adds.

For a 13 by 13 filter, the ideal operation-count speedup is 13 / 2 = 6.5 times.


In [ ]:
sigma = 2.0
size = int(2 * math.ceil(3 * sigma) + 1)
g1 = cv2.getGaussianKernel(size, sigma).reshape(-1)
g2 = np.outer(g1, g1)

direct = cv2.filter2D(gray.astype(np.float32), -1, g2, borderType=cv2.BORDER_REFLECT_101)
separable = cv2.sepFilter2D(gray.astype(np.float32), -1, g1, g1, borderType=cv2.BORDER_REFLECT_101)

M = 13
print("Maximum numerical difference:", np.max(np.abs(direct - separable)))
print(f"Ideal operation-count speedup for M={M}: {M**2 / (2*M):.1f}x")
show_images([to_uint8(direct), to_uint8(separable)], ["Direct 2D Gaussian", "Two separable 1D passes"], cols=2)


## 16. Why a weighted mean fails on an outlier

The lecture signal contains the outlier 1000. The filter [0.1, 0.8, 0.1] spreads its influence into neighboring outputs.


In [ ]:
outlier_signal = np.array([10, 12, 9, 8, 1000, 11, 10, 12], dtype=float)
weighted_kernel = np.array([0.1, 0.8, 0.1])
weighted_output = np.correlate(outlier_signal, weighted_kernel, mode="valid")

print("Signal:", outlier_signal)
print("Weighted mean output:", weighted_output)
print("Lecture values: [11.5, 9.2, 107.3, 801.9, 109.8, 10.3]")


## 17. Median filter calculations

The median resists isolated extreme values because it selects the middle sorted value instead of averaging all values.


In [ ]:
neighborhood_1 = np.array([40, 81, 13, 125, 830, 76, 144, 92, 108])
neighborhood_2 = np.array([830, 76, 80, 92, 108, 95, 102, 106, 87])

for number, values in enumerate([neighborhood_1, neighborhood_2], start=1):
    print(f"Neighborhood {number}: {values.tolist()}")
    print("Sorted:", np.sort(values).tolist())
    print("Median:", np.median(values), "\n")


### Median filtering is nonlinear

The lecture counterexample shows that median(A + B) does not equal median(A) + median(B).


In [ ]:
A = np.array([[1, 1, 1], [1, 1, 2], [2, 2, 2]])
B = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]])

median_A = np.median(A)
median_B = np.median(B)
median_sum = np.median(A + B)

print("median(A) =", median_A)
print("median(B) =", median_B)
print("median(A) + median(B) =", median_A + median_B)
print("median(A + B) =", median_sum)
print("Linear?", np.isclose(median_sum, median_A + median_B))


### Median filtering on salt-and-pepper noise


In [ ]:
sp_bgr = cv2.cvtColor(salt_pepper_noisy, cv2.COLOR_RGB2BGR)
median3 = cv2.cvtColor(cv2.medianBlur(sp_bgr, 3), cv2.COLOR_BGR2RGB)
median7 = cv2.cvtColor(cv2.medianBlur(sp_bgr, 7), cv2.COLOR_BGR2RGB)
gaussian_sp = cv2.GaussianBlur(salt_pepper_noisy, (7, 7), 1.5)

show_images(
    [salt_pepper_noisy, gaussian_sp, median3, median7],
    ["Salt-and-pepper input", "Gaussian filter", "Median size 3", "Median size 7"],
    cols=2,
)


## 18. Unsharp masking

Let details = image - smoothed. The sharpened image is image + alpha times details. The lecture compares alpha values 0, 1, 2, and 10.


In [ ]:
image_f = rgb.astype(np.float32)
smoothed = cv2.GaussianBlur(image_f, (0, 0), sigmaX=2.0)
details = image_f - smoothed

alphas = [0, 1, 2, 10]
unsharp_results = [to_uint8(image_f + alpha * details) for alpha in alphas]
show_images(unsharp_results, [f"Unsharp masking alpha={alpha}" for alpha in alphas], cols=2)


## 19. Image derivatives and edge strength

The lecture uses the derivative filters Dx = [-1, 0, 1] and Dy = Dx transpose. The gradient magnitude is sqrt(Dx^2 + Dy^2).


In [ ]:
lecture_image = np.array([
    [10, 10, 10, 50, 50, 50],
    [10, 10, 10, 50, 50, 50],
    [10, 10, 10, 50, 50, 50],
    [30, 30, 30, 70, 70, 70],
    [30, 30, 30, 70, 70, 70],
    [30, 30, 30, 70, 70, 70],
], dtype=float)

Dx = np.array([[-1, 0, 1]], dtype=float)
Dy = Dx.T
dx_valid = correlate2d(lecture_image, Dx, mode="valid")
dy_valid = correlate2d(lecture_image, Dy, mode="valid")

# Crop both valid responses to the common 4x4 interior before combining.
dx_common = dx_valid[1:-1, :]
dy_common = dy_valid[:, 1:-1]
gradient = np.sqrt(dx_common ** 2 + dy_common ** 2)

print("Original 6x6 image:\n", lecture_image.astype(int))
print("\nDx result, shape", dx_valid.shape, "\n", dx_valid)
print("\nDy result, shape", dy_valid.shape, "\n", dy_valid)
print("\nGradient magnitude, shape", gradient.shape, "\n", np.round(gradient, 1))


### Derivative filters on the uploaded image


In [ ]:
gray_f = gray.astype(np.float32)
dx = cv2.filter2D(gray_f, -1, Dx, borderType=cv2.BORDER_REFLECT_101)
dy = cv2.filter2D(gray_f, -1, Dy, borderType=cv2.BORDER_REFLECT_101)
gradient_image = np.sqrt(dx ** 2 + dy ** 2)

def normalize_for_display(image):
    image = np.abs(image).astype(float)
    if image.max() == 0:
        return np.zeros_like(image, dtype=np.uint8)
    return to_uint8(255 * image / image.max())

show_images(
    [gray, normalize_for_display(dx), normalize_for_display(dy), normalize_for_display(gradient_image)],
    ["Grayscale", "Horizontal derivative Dx", "Vertical derivative Dy", "Gradient magnitude"],
    cols=2,
)


## 20. Student experiment on the uploaded image

Change the Colab controls, then rerun this cell. This provides one place to compare padding, filter type, size, and strength.


In [ ]:
filter_choice = "Gaussian" #@param ["Box", "Gaussian", "Median", "Sharpen", "Horizontal derivative", "Vertical derivative"]
kernel_size = 5 #@param {type:"slider", min:3, max:21, step:2}
sigma = 1.5 #@param {type:"slider", min:0.5, max:8.0, step:0.5}
sharpen_alpha = 1.5 #@param {type:"slider", min:0.0, max:10.0, step:0.5}
padding_choice = "Reflect" #@param ["Zero", "Reflect", "Symmetric", "Circular"]

if kernel_size % 2 == 0:
    kernel_size += 1

mode_map = {
    "Zero": "constant",
    "Reflect": "mirror",
    "Symmetric": "reflect",
    "Circular": "wrap",
}
mode = mode_map[padding_choice]

if filter_choice == "Box":
    kernel = np.ones((kernel_size, kernel_size), dtype=float) / kernel_size ** 2
    result = to_uint8(np.stack([ndimage.correlate(rgb[:, :, c].astype(float), kernel, mode=mode, cval=0) for c in range(3)], axis=2))
elif filter_choice == "Gaussian":
    radius = (kernel_size - 1) / 2
    truncate = radius / sigma
    result = to_uint8(np.stack([ndimage.gaussian_filter(rgb[:, :, c].astype(float), sigma=sigma, mode=mode, cval=0, truncate=truncate) for c in range(3)], axis=2))
elif filter_choice == "Median":
    result = np.stack([ndimage.median_filter(rgb[:, :, c], size=kernel_size, mode=mode, cval=0) for c in range(3)], axis=2)
elif filter_choice == "Sharpen":
    radius = (kernel_size - 1) / 2
    truncate = radius / sigma
    smooth = np.stack([ndimage.gaussian_filter(rgb[:, :, c].astype(float), sigma=sigma, mode=mode, cval=0, truncate=truncate) for c in range(3)], axis=2)
    result = to_uint8(rgb.astype(np.float32) + sharpen_alpha * (rgb.astype(np.float32) - smooth))
elif filter_choice == "Horizontal derivative":
    result = normalize_for_display(ndimage.correlate(gray.astype(float), Dx, mode=mode, cval=0))
else:
    result = normalize_for_display(ndimage.correlate(gray.astype(float), Dy, mode=mode, cval=0))

show_images([rgb, result], ["Uploaded image", f"{filter_choice}, padding={padding_choice}"], cols=2)


# Practice tasks

1. Choose one pixel away from the boundary. Write the 3 by 3 neighborhood and calculate the box-filter output by hand. Compare it with the notebook.
2. Repeat the calculation for a corner pixel using zero, symmetric, and circular padding.
3. Add Gaussian noise and salt-and-pepper noise. Compare Gaussian and median filtering for each noise type.
4. Compare Gaussian sigma values 1, 3, and 7. Explain the change in detail and edge sharpness.
5. Compare unsharp masking with alpha values 1, 2, and 10. Identify clipping or artifacts.
6. Replace the derivative kernel with a Sobel kernel and compare the edge map.
7. Verify one convolution property numerically with two small kernels.


# Recap

- Sampling controls spatial resolution. Quantization controls the number of available intensity values.
- Point processing uses one input pixel at a time. Spatial filtering uses a neighborhood.
- Linear filtering calculates a weighted sum. Padding determines what happens near image boundaries.
- Correlation preserves kernel orientation. Convolution flips the kernel first.
- Box and Gaussian filters smooth images. A Gaussian filter is separable and computationally efficient.
- Median filtering removes impulse noise well and is nonlinear.
- Sharpening adds scaled image detail. Derivative filters reveal intensity changes and edges.
